# MLE vs Bayesian Parameter Estimation — and What It Does to a Classifier

**Goal:** compare *maximum likelihood estimation* (MLE) with *Bayesian parameter estimation* on **simulated data**
(so we know the true parameters), then analyse how each choice changes the behaviour of a Gaussian classifier.

**How to run:** `Runtime → Run all`. The full run takes about 3–4 minutes on Colab. Set `QUICK = True` below to get a ~30 s preview.

Notebook outline:
1. Setup
2. Estimators: MLE and Bayesian (Normal-Inverse-Wishart prior)
3. Simulated data
4. Evaluation metrics
5. Experiment 1: pure parameter estimation
6. Experiment 2: classifier learning curves (2-D and 10-D)
7. Experiment 3: decision boundaries
8. Experiment 4: calibration
9. Experiment 5: prior sensitivity
10. Conclusions

In [ ]:
# Colab already has numpy, scipy and matplotlib installed.
import os, time, numpy as np
QUICK = False          # True = fewer Monte-Carlo trials (fast preview)
OUT = "results"        # figures are also saved here (see the Files panel on the left)
os.makedirs(OUT, exist_ok=True)
T = lambda full, quick: quick if QUICK else full   # helper to pick trial counts

## 2. The estimators

**MLE** picks the single parameter value that makes the data most likely:
$\hat\theta_{MLE} = \arg\max_\theta p(D\mid\theta)$. For a Gaussian class this gives
$\hat\mu = \bar x$ and $\hat\Sigma = \frac1n\sum_i (x_i-\bar x)(x_i-\bar x)^\top$.
The classifier then **plugs these in** as if they were exact.

**Bayesian estimation** treats the parameters as random. It puts a prior on them and computes the posterior
$p(\theta\mid D)\propto p(D\mid\theta)\,p(\theta)$.
We use the conjugate **Normal-Inverse-Wishart (NIW)** prior:
$\Sigma\sim IW(\Psi_0,\nu_0)$, $\mu\mid\Sigma\sim N(\mu_0,\Sigma/\kappa_0)$. The posterior is again NIW:

$$\kappa_n=\kappa_0+n,\quad \nu_n=\nu_0+n,\quad \mu_n=\frac{\kappa_0\mu_0+n\bar x}{\kappa_n},\quad
\Psi_n=\Psi_0+S+\tfrac{\kappa_0 n}{\kappa_n}(\bar x-\mu_0)(\bar x-\mu_0)^\top$$

To classify, the Bayesian model **averages over all plausible parameters** using the *posterior predictive*
$p(x\mid D)=\int p(x\mid\theta)p(\theta\mid D)\,d\theta$. Here that works out to a **multivariate Student-t**,
which has heavier tails when data are scarce. That is what stops it from being over-confident.

Class priors: MLE uses $n_k/n$. Bayes uses a Dirichlet(1) prior, giving $(n_k+1)/(n+K)$ (Laplace smoothing).

There are three classifiers:
* `MLEGaussianClassifier`: plug-in QDA
* `BayesGaussianClassifier`: the full posterior predictive (Student-t)
* `BayesMAPPlugInClassifier`: an *ablation* that plugs the Bayesian posterior-mean estimates into a Gaussian.
  It tells us how much of the gain comes from the prior and how much from averaging over uncertainty.

In [ ]:
import numpy as np
from scipy.special import gammaln, logsumexp


# ----------------------------------------------------------------------------- densities
def mvn_logpdf(X, mu, Sigma):
    """Log density of a multivariate normal, evaluated row-wise on X."""
    d = X.shape[1]
    L = np.linalg.cholesky(Sigma)
    z = np.linalg.solve(L, (X - mu).T)                 # (d, n)
    maha = np.sum(z ** 2, axis=0)
    logdet = 2.0 * np.sum(np.log(np.diag(L)))
    return -0.5 * (d * np.log(2 * np.pi) + logdet + maha)


def mvt_logpdf(X, mu, Scale, df):
    """Log density of a multivariate Student-t (location mu, scale matrix Scale, df)."""
    d = X.shape[1]
    L = np.linalg.cholesky(Scale)
    z = np.linalg.solve(L, (X - mu).T)
    maha = np.sum(z ** 2, axis=0)
    logdet = 2.0 * np.sum(np.log(np.diag(L)))
    return (gammaln((df + d) / 2) - gammaln(df / 2)
            - 0.5 * d * np.log(df * np.pi) - 0.5 * logdet
            - 0.5 * (df + d) * np.log1p(maha / df))


# ------------------------------------------------------------------- single-class estimators
def mle_gaussian(X, ridge=1e-6):
    """MLE of (mu, Sigma). Sigma uses 1/n (biased). A tiny ridge keeps it invertible
    when n <= d; without it MLE QDA is undefined at very small sample sizes."""
    n, d = X.shape
    mu = X.mean(axis=0)
    Xc = X - mu
    Sigma = Xc.T @ Xc / n + ridge * np.eye(d)
    return mu, Sigma


class NIWPrior:
    """Normal-Inverse-Wishart prior:  Sigma ~ IW(Psi0, nu0),  mu | Sigma ~ N(mu0, Sigma/kappa0)."""

    def __init__(self, d, mu0=None, kappa0=0.01, nu0=None, prior_cov=None):
        self.d = d
        self.mu0 = np.zeros(d) if mu0 is None else np.asarray(mu0, float)
        self.kappa0 = kappa0
        self.nu0 = d + 2 if nu0 is None else nu0           # smallest nu with a finite mean of Sigma
        prior_cov = np.eye(d) if prior_cov is None else np.asarray(prior_cov, float)
        # choose Psi0 so that the prior mean E[Sigma] = Psi0/(nu0-d-1) equals prior_cov
        self.Psi0 = prior_cov * (self.nu0 - d - 1)

    def posterior(self, X):
        n = X.shape[0]
        xbar = X.mean(axis=0)
        Xc = X - xbar
        S = Xc.T @ Xc
        kn = self.kappa0 + n
        nun = self.nu0 + n
        mun = (self.kappa0 * self.mu0 + n * xbar) / kn
        diff = (xbar - self.mu0)[:, None]
        Psin = self.Psi0 + S + (self.kappa0 * n / kn) * (diff @ diff.T)
        return NIWPosterior(mun, kn, nun, Psin)


class NIWPosterior:
    def __init__(self, mu, kappa, nu, Psi):
        self.mu, self.kappa, self.nu, self.Psi = mu, kappa, nu, Psi
        self.d = len(mu)

    @property
    def mean_mu(self):
        return self.mu

    @property
    def mean_Sigma(self):
        return self.Psi / (self.nu - self.d - 1)

    def predictive_params(self):
        """Posterior predictive p(x|D) = Student-t(df, mu_n, Scale)."""
        df = self.nu - self.d + 1
        Scale = self.Psi * (self.kappa + 1) / (self.kappa * df)
        return self.mu, Scale, df


# ------------------------------------------------------------------------------ classifiers
class MLEGaussianClassifier:
    name = "MLE (plug-in QDA)"

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n = len(y)
        self.params_ = [mle_gaussian(X[y == c]) for c in self.classes_]
        self.log_prior_ = np.log([np.sum(y == c) / n for c in self.classes_])
        return self

    def log_joint(self, X):
        return np.column_stack([lp + mvn_logpdf(X, mu, S)
                                for lp, (mu, S) in zip(self.log_prior_, self.params_)])

    def predict_proba(self, X):
        lj = self.log_joint(X)
        return np.exp(lj - logsumexp(lj, axis=1, keepdims=True))

    def predict(self, X):
        return self.classes_[np.argmax(self.log_joint(X), axis=1)]


class BayesGaussianClassifier(MLEGaussianClassifier):
    name = "Bayesian (posterior predictive)"

    def __init__(self, kappa0=0.01, nu0=None, prior_cov=None, mu0=None, alpha=1.0):
        self.kappa0, self.nu0, self.prior_cov, self.mu0, self.alpha = kappa0, nu0, prior_cov, mu0, alpha

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        d = X.shape[1]
        # mu0 may be one vector shared by all classes, or one row per class
        mu0 = None if self.mu0 is None else np.asarray(self.mu0, float)
        per_class = mu0 is not None and mu0.ndim == 2
        self.posteriors_ = []
        for k, c in enumerate(self.classes_):
            m0 = mu0[k] if per_class else mu0
            prior = NIWPrior(d, mu0=m0, kappa0=self.kappa0, nu0=self.nu0, prior_cov=self.prior_cov)
            self.posteriors_.append(prior.posterior(X[y == c]))
        counts = np.array([np.sum(y == c) for c in self.classes_], float)
        K = len(self.classes_)
        # Dirichlet(alpha) posterior predictive for the class label
        self.log_prior_ = np.log((counts + self.alpha) / (counts.sum() + K * self.alpha))
        return self

    def log_joint(self, X):
        cols = []
        for lp, post in zip(self.log_prior_, self.posteriors_):
            mu, Scale, df = post.predictive_params()
            cols.append(lp + mvt_logpdf(X, mu, Scale, df))
        return np.column_stack(cols)


class BayesMAPPlugInClassifier(BayesGaussianClassifier):
    """Ablation: Bayesian *posterior-mean* parameters plugged into a Gaussian.
    Separates the effect of the prior (shrinkage) from that of predictive averaging."""
    name = "Bayes posterior-mean plug-in"

    def log_joint(self, X):
        return np.column_stack([lp + mvn_logpdf(X, p.mean_mu, p.mean_Sigma)
                                for lp, p in zip(self.log_prior_, self.posteriors_)])

## 3. Simulated data

* **2-D problem:** class 0 ~ N((0,0), [[1,.5],[.5,1]]), class 1 ~ N((1.5,1), [[1.5,−.4],[−.4,.8]]).
  The covariances differ, so the best possible boundary is curved (quadratic), and the best possible accuracy is about 0.82.
* **10-D problem:** random correlated covariances. Used to show what happens when n is close to the dimension d.

Because we know the true parameters, `bayes_optimal_proba` gives the **best any classifier could ever do**.
That is our reference line.

In [ ]:
import numpy as np


class GaussianMixtureProblem:
    def __init__(self, means, covs, priors, seed=None):
        self.means = [np.asarray(m, float) for m in means]
        self.covs = [np.asarray(c, float) for c in covs]
        self.priors = np.asarray(priors, float)
        self.d = len(self.means[0])
        self.rng = np.random.default_rng(seed)

    def sample_per_class(self, n_per_class):
        X, y = [], []
        for k, (m, c) in enumerate(zip(self.means, self.covs)):
            X.append(self.rng.multivariate_normal(m, c, size=n_per_class))
            y.append(np.full(n_per_class, k))
        return np.vstack(X), np.concatenate(y)

    def sample(self, n):
        y = self.rng.choice(len(self.priors), size=n, p=self.priors)
        X = np.empty((n, self.d))
        for k in range(len(self.priors)):
            idx = y == k
            X[idx] = self.rng.multivariate_normal(self.means[k], self.covs[k], size=idx.sum())
        return X, y

    def bayes_optimal_proba(self, X):
        """True posterior P(y|x) — the best any classifier can do."""
        from scipy.special import logsumexp
        lj = np.column_stack([np.log(p) + mvn_logpdf(X, m, c)
                              for p, m, c in zip(self.priors, self.means, self.covs)])
        return np.exp(lj - logsumexp(lj, axis=1, keepdims=True))


def problem_2d(seed=0):
    """Two overlapping 2-D classes with different covariances (quadratic Bayes boundary)."""
    return GaussianMixtureProblem(
        means=[[0.0, 0.0], [1.5, 1.0]],
        covs=[[[1.0, 0.5], [0.5, 1.0]], [[1.5, -0.4], [-0.4, 0.8]]],
        priors=[0.5, 0.5], seed=seed)


def problem_highdim(d=10, seed=0):
    """d-dimensional problem where the class difference is spread across all dimensions."""
    rng = np.random.default_rng(1234)
    A = rng.normal(size=(d, d)) / np.sqrt(d)
    cov0 = np.eye(d) + 0.3 * (A @ A.T)
    B = rng.normal(size=(d, d)) / np.sqrt(d)
    cov1 = np.eye(d) + 0.3 * (B @ B.T)
    mean1 = np.full(d, 1.0 / np.sqrt(d)) * 1.6
    return GaussianMixtureProblem(means=[np.zeros(d), mean1], covs=[cov0, cov1],
                                  priors=[0.5, 0.5], seed=seed)

In [ ]:
# Quick look at the simulated data
import matplotlib.pyplot as plt
prob = problem_2d(seed=0)
X, y = prob.sample(1000)
plt.figure(figsize=(5,4))
plt.scatter(*X[y==0].T, s=6, label="class 0"); plt.scatter(*X[y==1].T, s=6, label="class 1")
plt.legend(); plt.title("Simulated 2-D data (1000 points)"); plt.show()

## 4. Metrics

* **Accuracy:** fraction correct.
* **Log-loss:** $-\frac1N\sum\log p(y_i\mid x_i)$. It heavily punishes confident wrong answers.
* **Brier score:** squared error of the predicted probabilities.
* **ECE (expected calibration error):** does "80 % confident" really mean right 80 % of the time? 0 = perfectly calibrated.

In [ ]:
import numpy as np


def accuracy(y, proba):
    return np.mean(np.argmax(proba, axis=1) == y)


def log_loss(y, proba, eps=1e-15):
    p = np.clip(proba[np.arange(len(y)), y], eps, 1.0)
    return -np.mean(np.log(p))


def brier(y, proba):
    onehot = np.eye(proba.shape[1])[y]
    return np.mean(np.sum((proba - onehot) ** 2, axis=1))


def ece(y, proba, n_bins=10):
    """Expected calibration error on the confidence of the predicted class."""
    conf = proba.max(axis=1)
    correct = (np.argmax(proba, axis=1) == y).astype(float)
    bins = np.linspace(0.5 if proba.shape[1] == 2 else 0.0, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    total = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            total += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return total


def reliability_curve(y, proba, n_bins=10):
    """Fraction of positives vs mean predicted P(class 1), for a reliability diagram."""
    p1 = proba[:, 1]
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p1, bins) - 1, 0, n_bins - 1)
    mean_pred, frac_pos = [], []
    for b in range(n_bins):
        m = idx == b
        if m.sum() > 20:
            mean_pred.append(p1[m].mean())
            frac_pos.append((y[m] == 1).mean())
    return np.array(mean_pred), np.array(frac_pos)


ALL_METRICS = {"accuracy": accuracy, "log_loss": log_loss, "brier": brier, "ece": ece}

In [ ]:
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats

COLORS = {"MLE": "#d1495b", "Bayes": "#00798c", "BayesPlugIn": "#edae49", "Optimal": "#555555"}
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})


def _models(**bayes_kw):
    return {"MLE": MLEGaussianClassifier(),
            "Bayes": BayesGaussianClassifier(**bayes_kw),
            "BayesPlugIn": BayesMAPPlugInClassifier(**bayes_kw)}

## 5. Experiment 1: pure parameter estimation

Before any classifier, how close do the estimates get to the truth? We repeat each sample size thousands of times
and measure the mean squared error (MSE) for:
* a coin's probability *p* (Beta(2,2) prior)
* a Gaussian mean (N(0, 1.5²) prior)
* a 2-D covariance matrix (NIW prior)

**What to look for:** at small *n*, the Bayesian posterior mean has lower MSE. It is slightly biased toward the prior
but has much lower variance (the **bias–variance trade-off**). As *n* grows, the curves merge.

In [ ]:
def exp_parameter_estimation(out, n_trials=2000, seed=0):
    rng = np.random.default_rng(seed)
    ns = np.array([1, 2, 3, 5, 8, 12, 20, 35, 60, 100, 200])
    res = {"n": ns.tolist()}

    # (a) Bernoulli p with Beta(a,b) prior
    p_true, a, b = 0.7, 2.0, 2.0
    mse_mle, mse_bayes = [], []
    for n in ns:
        k = rng.binomial(n, p_true, size=n_trials)
        mse_mle.append(np.mean((k / n - p_true) ** 2))
        mse_bayes.append(np.mean(((k + a) / (n + a + b) - p_true) ** 2))
    res["bernoulli"] = {"mle": mse_mle, "bayes": mse_bayes}

    # (b) Gaussian mean, known sigma, N(mu0, tau^2) prior — plus 95% interval coverage
    mu_true, sigma, mu0, tau = 1.0, 2.0, 0.0, 1.5
    mse_mle_g, mse_bayes_g, cov_mle, cov_bayes = [], [], [], []
    for n in ns:
        x = rng.normal(mu_true, sigma, size=(n_trials, n))
        xbar = x.mean(axis=1)
        prec = 1 / tau ** 2 + n / sigma ** 2
        post_mean = (mu0 / tau ** 2 + n * xbar / sigma ** 2) / prec
        post_sd = np.sqrt(1 / prec)
        mse_mle_g.append(np.mean((xbar - mu_true) ** 2))
        mse_bayes_g.append(np.mean((post_mean - mu_true) ** 2))
        se = sigma / np.sqrt(n)
        cov_mle.append(np.mean(np.abs(xbar - mu_true) < 1.96 * se))
        cov_bayes.append(np.mean(np.abs(post_mean - mu_true) < 1.96 * post_sd))
    res["gauss_mean"] = {"mle": mse_mle_g, "bayes": mse_bayes_g,
                         "coverage_mle": cov_mle, "coverage_bayes": cov_bayes}

    # (c) 2-D covariance matrix (Frobenius error) — MLE vs NIW posterior mean
    prob = problem_2d(seed=seed)
    Sig_true = prob.covs[0]
    prior = NIWPrior(2, kappa0=0.01)
    ns_c = ns[ns >= 3]
    fe_mle, fe_bayes = [], []
    for n in ns_c:
        e1, e2 = [], []
        for _ in range(n_trials // 4):
            X = rng.multivariate_normal(prob.means[0], Sig_true, size=n)
            e1.append(np.linalg.norm(mle_gaussian(X, ridge=0)[1] - Sig_true))
            e2.append(np.linalg.norm(prior.posterior(X).mean_Sigma - Sig_true))
        fe_mle.append(np.mean(np.square(e1)))
        fe_bayes.append(np.mean(np.square(e2)))
    res["covariance"] = {"n": ns_c.tolist(), "mle": fe_mle, "bayes": fe_bayes}

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
    ax[0].loglog(ns, mse_mle, "o-", c=COLORS["MLE"], label="MLE  k/n")
    ax[0].loglog(ns, mse_bayes, "s-", c=COLORS["Bayes"], label="Posterior mean (Beta(2,2) prior)")
    ax[0].set(title=f"Bernoulli p (true p={p_true})", xlabel="n", ylabel="MSE")
    ax[1].loglog(ns, mse_mle_g, "o-", c=COLORS["MLE"], label="MLE  x̄")
    ax[1].loglog(ns, mse_bayes_g, "s-", c=COLORS["Bayes"], label=f"Posterior mean (N({mu0},{tau}²) prior)")
    ax[1].loglog(ns, sigma ** 2 / ns, "k:", lw=1, label="σ²/n (Cramér–Rao bound)")
    ax[1].set(title=f"Gaussian mean (true μ={mu_true}, σ={sigma})", xlabel="n", ylabel="MSE")
    ax[2].loglog(ns_c, fe_mle, "o-", c=COLORS["MLE"], label="MLE  S/n")
    ax[2].loglog(ns_c, fe_bayes, "s-", c=COLORS["Bayes"], label="NIW posterior mean")
    ax[2].set(title="2-D covariance matrix", xlabel="n", ylabel="E‖Σ̂ − Σ‖²_F")
    for a_ in ax:
        a_.legend(fontsize=8)
    fig.suptitle("Parameter estimation error: MLE vs Bayesian estimates", fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(out, "01_parameter_estimation_mse.png"))
    plt.show()

    # posterior evolution for the coin — how belief concentrates as data arrive
    fig, ax = plt.subplots(figsize=(7, 4.2))
    xs = np.linspace(0, 1, 400)
    flips = rng.random(200) < p_true
    for n, shade in zip([0, 1, 5, 20, 200], np.linspace(0.25, 1, 5)):
        k = flips[:n].sum()
        ax.plot(xs, stats.beta.pdf(xs, a + k, b + n - k), c=plt.cm.viridis(shade),
                label=f"n={n}, heads={k}" + (f", MLE={k / n:.2f}" if n else ""))
    ax.axvline(p_true, c="k", ls="--", lw=1, label="true p")
    ax.set(title="Posterior over p sharpens as data arrives (Beta(2,2) prior)",
           xlabel="p", ylabel="posterior density")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "02_posterior_evolution.png"))
    plt.show()
    return res

In [ ]:
pe = exp_parameter_estimation(OUT, n_trials=T(2000, 400))
i = pe["n"].index(5)
print(f"n=5  Bernoulli MSE   MLE={pe['bernoulli']['mle'][i]:.4f}  Bayes={pe['bernoulli']['bayes'][i]:.4f}")
print(f"n=5  Gauss-mean MSE  MLE={pe['gauss_mean']['mle'][i]:.4f}  Bayes={pe['gauss_mean']['bayes'][i]:.4f}")

## 6. Experiment 2: classifier learning curves

For each training size (points per class), we train all three classifiers on 200 fresh random training sets,
then test each on 20 000 new points. Shaded bands are 95 % confidence intervals (IQR for log-loss).
The dotted line is the Bayes-optimal classifier.

**What to look for:**
* Bayes is much more accurate at tiny *n*.
* MLE's log-loss and calibration error explode at small *n*, because it is **over-confident**.
* The yellow ablation gets most of the *accuracy* gain but not all of the *calibration* gain.
  So the prior fixes the boundary, and the predictive averaging fixes the probabilities.
* Everything converges as *n* grows.

In [ ]:
def _learning_curves(problem_fn, ns, n_trials, n_test, seed, **bayes_kw):
    prob = problem_fn(seed=seed)
    Xte, yte = prob.sample(n_test)
    opt = prob.bayes_optimal_proba(Xte)
    optimal = {m: f(yte, opt) for m, f in ALL_METRICS.items()}
    names = list(_models().keys())
    raw = {nm: {m: np.zeros((len(ns), n_trials)) for m in ALL_METRICS} for nm in names}
    for i, n in enumerate(ns):
        for t in range(n_trials):
            Xtr, ytr = prob.sample_per_class(n)
            for nm, model in _models(**bayes_kw).items():
                p = model.fit(Xtr, ytr).predict_proba(Xte)
                for m, f in ALL_METRICS.items():
                    raw[nm][m][i, t] = f(yte, p)
    return raw, optimal


def _plot_curves(raw, optimal, ns, title, path):
    labels = {"MLE": "MLE (plug-in)", "Bayes": "Bayesian (posterior predictive)",
              "BayesPlugIn": "Bayes posterior-mean plug-in"}
    titles = {"accuracy": "Test accuracy  ↑", "log_loss": "Test log-loss  ↓",
              "brier": "Brier score  ↓", "ece": "Expected calibration error  ↓"}
    fig, axes = plt.subplots(1, 4, figsize=(19, 4.3))
    for ax, m in zip(axes, ALL_METRICS):
        for nm, r in raw.items():
            vals = r[m]
            if m == "log_loss":
                med = np.median(vals, axis=1)        # MLE log-loss has heavy outliers at tiny n
                lo, hi = np.percentile(vals, [25, 75], axis=1)
            else:
                med = vals.mean(axis=1)
                se = vals.std(axis=1) / np.sqrt(vals.shape[1])
                lo, hi = med - 1.96 * se, med + 1.96 * se
            ls = "--" if nm == "BayesPlugIn" else "-"
            ax.plot(ns, med, ls, marker="o", ms=4, c=COLORS[nm], label=labels[nm])
            ax.fill_between(ns, lo, hi, color=COLORS[nm], alpha=0.15)
        if m != "ece":
            ax.axhline(optimal[m], c=COLORS["Optimal"], ls=":", label="Bayes-optimal (true params)")
        ax.set_xscale("log")
        ax.set(title=titles[m] + (" (median, IQR)" if m == "log_loss" else ""),
               xlabel="training samples per class")
        if m == "log_loss":
            ax.set_ylim(optimal[m] * 0.95, min(ax.get_ylim()[1], optimal[m] * 2.2))
    axes[0].legend(fontsize=8)
    fig.suptitle(title, fontweight="bold")
    fig.tight_layout()
    fig.savefig(path)
    plt.show()


def _summarise(raw, optimal, ns):
    rows = []
    for i, n in enumerate(ns):
        for nm, r in raw.items():
            row = {"n_per_class": int(n), "model": nm}
            for m in ALL_METRICS:
                row[m] = float(r[m][i].mean())
                row[m + "_median"] = float(np.median(r[m][i]))
                row[m + "_std"] = float(r[m][i].std())
            rows.append(row)
    return {"rows": rows, "optimal": optimal}


def exp_learning_curves_2d(out, n_trials=200, n_test=20000, seed=0):
    ns = np.array([3, 4, 5, 8, 12, 20, 35, 60, 100, 200, 500])
    raw, opt = _learning_curves(problem_2d, ns, n_trials, n_test, seed)
    _plot_curves(raw, opt, ns, "2-D Gaussian classes: classifier performance vs training size",
                 os.path.join(out, "03_learning_curves_2d.png"))
    return _summarise(raw, opt, ns)


def exp_learning_curves_highdim(out, d=10, n_trials=100, n_test=20000, seed=0):
    ns = np.array([d + 1, d + 3, 15, 20, 30, 50, 80, 150, 300, 600])
    raw, opt = _learning_curves(lambda seed: problem_highdim(d=d, seed=seed), ns,
                                n_trials, n_test, seed)
    _plot_curves(raw, opt, ns, f"{d}-D Gaussian classes: where MLE struggles (n close to d)",
                 os.path.join(out, "04_learning_curves_highdim.png"))
    return _summarise(raw, opt, ns)

In [ ]:
import pandas as pd
lc = exp_learning_curves_2d(OUT, n_trials=T(200, 30), n_test=T(20000, 5000))
df = pd.DataFrame(lc["rows"])
df[df.n_per_class.isin([3,5,12,35,100,500])].pivot(index="n_per_class", columns="model",
                                                  values=["accuracy","log_loss_median","ece"]).round(3)

### Same thing in 10 dimensions
With d = 10, a covariance matrix has 55 free numbers. With only ~11 points per class, the MLE covariance is nearly
singular, and the MLE classifier becomes wildly over-confident. The Bayesian prior acts as a regulariser.

In [ ]:
hd = exp_learning_curves_highdim(OUT, n_trials=T(100, 15), n_test=T(20000, 5000))
dh = pd.DataFrame(hd["rows"])
dh[dh.n_per_class.isin([11,15,30,80,300,600])].pivot(index="n_per_class", columns="model",
                                                    values=["accuracy","log_loss_median","ece"]).round(3)

## 7. Experiment 3: decision boundaries

Colour = predicted P(class 1 | x). Black = learned boundary, green dashed = true optimal boundary,
dotted = 10 % / 90 % confidence contours.

**What to look for:** with 4 points, MLE draws a sharp boundary and is ~100 % sure almost everywhere,
even far from any data. Bayes produces soft, pale regions where it is unsure. With 200 points the two are almost identical.

In [ ]:
def exp_decision_boundaries(out, seed=3):
    prob = problem_2d(seed=seed)
    gx, gy = np.meshgrid(np.linspace(-4, 5.5, 300), np.linspace(-3.5, 4.5, 300))
    G = np.column_stack([gx.ravel(), gy.ravel()])
    p_opt = prob.bayes_optimal_proba(G)[:, 1].reshape(gx.shape)
    sizes = [4, 15, 200]
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for j, n in enumerate(sizes):
        Xtr, ytr = prob.sample_per_class(n)
        for i, model in enumerate([MLEGaussianClassifier(), BayesGaussianClassifier()]):
            ax = axes[i, j]
            p = model.fit(Xtr, ytr).predict_proba(G)[:, 1].reshape(gx.shape)
            cf = ax.contourf(gx, gy, p, levels=np.linspace(0, 1, 21), cmap="RdBu_r", alpha=0.75)
            ax.contour(gx, gy, p, levels=[0.5], colors="k", linewidths=2)
            ax.contour(gx, gy, p_opt, levels=[0.5], colors="lime", linewidths=1.5, linestyles="--")
            ax.contour(gx, gy, p, levels=[0.1, 0.9], colors="k", linewidths=0.6, linestyles=":")
            ax.scatter(*Xtr[ytr == 0].T, c="#1f4e9c", edgecolor="w", s=28 if n < 50 else 8)
            ax.scatter(*Xtr[ytr == 1].T, c="#b2182b", edgecolor="w", s=28 if n < 50 else 8)
            ax.set_title(f"{model.name}\nn = {n} per class", fontsize=10)
            ax.grid(False)
    fig.colorbar(cf, ax=axes, shrink=0.6, label="P(class 1 | x)")
    fig.suptitle("Decision boundaries (black = learned P=0.5, dotted = P=0.1/0.9, "
                 "green dashed = Bayes-optimal)", fontweight="bold")
    fig.savefig(os.path.join(out, "05_decision_boundaries.png"), bbox_inches="tight")
    plt.show()

In [ ]:
exp_decision_boundaries(OUT)

## 8. Experiment 4: calibration at n = 5 per class

**Reliability diagram:** if the model says 70 %, it should be right 70 % of the time (points on the diagonal).
MLE's curve is flatter than the diagonal: its probabilities are too extreme. The histogram shows MLE piling up
near confidence 1.0.

In [ ]:
def exp_calibration(out, n_small=5, n_trials=300, n_test=4000, seed=1):
    prob = problem_2d(seed=seed)
    Xte, yte = prob.sample(n_test)
    preds = {"MLE": [], "Bayes": []}
    for _ in range(n_trials):
        Xtr, ytr = prob.sample_per_class(n_small)
        preds["MLE"].append(MLEGaussianClassifier().fit(Xtr, ytr).predict_proba(Xte))
        preds["Bayes"].append(BayesGaussianClassifier().fit(Xtr, ytr).predict_proba(Xte))
    y_all = np.tile(yte, n_trials)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
    ax[0].plot([0, 1], [0, 1], "k:", label="perfectly calibrated")
    res = {}
    for nm, label in [("MLE", "MLE (plug-in)"), ("Bayes", "Bayesian (posterior predictive)")]:
        P = np.vstack(preds[nm])
        mp, fp = reliability_curve(y_all, P, n_bins=15)
        ax[0].plot(mp, fp, "o-", c=COLORS[nm], label=label)
        conf = P.max(axis=1)
        ax[1].hist(conf, bins=30, range=(0.5, 1), alpha=0.55, color=COLORS[nm], label=label, density=True)
        res[nm] = {"mean_confidence": float(conf.mean()),
                   "accuracy": float((P.argmax(1) == y_all).mean()),
                   "frac_conf_above_0.99": float((conf > 0.99).mean())}
    ax[0].set(title=f"Reliability diagram (n = {n_small} per class, {n_trials} trainings)",
              xlabel="predicted P(class 1)", ylabel="observed frequency of class 1")
    ax[1].set(title="Distribution of prediction confidence", xlabel="max predicted probability",
              ylabel="density")
    for a in ax:
        a.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "06_calibration_small_n.png"))
    plt.show()
    return res

In [ ]:
cal = exp_calibration(OUT, n_trials=T(300, 60))
pd.DataFrame(cal).T.round(3)

## 9. Experiment 5: prior sensitivity

$\kappa_0$ acts like "the number of imaginary data points" backing the prior mean. We try a **correct** prior
(the true class means) and a **wrong** one (the two classes' means swapped), at several strengths.

**What to look for:** a strong correct prior is excellent with tiny data. A strong wrong prior is *worse than guessing*
until enough real data outvote it. A weak prior (κ₀ ≤ 1) is safe either way.

In [ ]:
def exp_prior_sensitivity(out, n_trials=150, n_test=10000, seed=2):
    """Vary prior strength kappa0 (= number of 'pseudo-observations' backing the prior mean)
    with a per-class prior mean that is either CORRECT (true class means) or WRONG
    (the two classes' means swapped)."""
    prob = problem_2d(seed=seed)
    Xte, yte = prob.sample(n_test)
    ns = np.array([3, 5, 10, 20, 50, 100, 300])
    kappas = [0.01, 1, 10, 50]
    true_means = np.array(prob.means)
    settings = {"correct prior: μ₀ₖ = true class means": true_means,
                "wrong prior: μ₀ₖ = the OTHER class's mean": true_means[::-1].copy()}
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
    res = {}
    for ax, (title, mu0) in zip(axes, settings.items()):
        mle_acc = []
        for n in ns:
            accs = []
            for _ in range(n_trials):
                Xtr, ytr = prob.sample_per_class(n)
                accs.append(ALL_METRICS["accuracy"](yte, MLEGaussianClassifier().fit(Xtr, ytr).predict_proba(Xte)))
            mle_acc.append(np.mean(accs))
        ax.plot(ns, mle_acc, "o-", c=COLORS["MLE"], lw=2.5, label="MLE")
        res[title] = {"MLE": mle_acc}
        for kc, col in zip(kappas, plt.cm.viridis(np.linspace(0.1, 0.85, len(kappas)))):
            acc = []
            for n in ns:
                accs = []
                for _ in range(n_trials):
                    Xtr, ytr = prob.sample_per_class(n)
                    m = BayesGaussianClassifier(kappa0=kc, mu0=mu0)
                    accs.append(ALL_METRICS["accuracy"](yte, m.fit(Xtr, ytr).predict_proba(Xte)))
                acc.append(np.mean(accs))
            ax.plot(ns, acc, "s--", c=col, label=f"Bayes κ₀={kc}")
            res[title][f"kappa0={kc}"] = acc
        ax.set_xscale("log")
        ax.set(title=title, xlabel="training samples per class", ylabel="test accuracy")
        ax.legend(fontsize=8)
    fig.suptitle("Prior sensitivity: a strong prior helps if right, hurts if wrong; "
                 "either way the data eventually win", fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(out, "07_prior_sensitivity.png"))
    plt.show()
    res["n"] = ns.tolist()
    return res

In [ ]:
ps = exp_prior_sensitivity(OUT, n_trials=T(150, 25))
pd.concat({k: pd.DataFrame(v, index=ps["n"]).T for k, v in ps.items() if k != "n"}).round(3)

## 10. Conclusions

| | MLE | Bayesian |
|---|---|---|
| Output | a single point estimate | a full posterior distribution |
| Small *n* | high variance, over-confident, can be singular | regularised, better calibrated |
| Large *n* | consistent and efficient | converges to MLE (the prior washes out) |
| Requires | nothing extra | a prior, which can hurt if it is strong and wrong |
| Cost | closed form | closed form here (conjugate); in general MCMC / variational inference |

1. With little data, Bayesian estimation gives better parameter estimates (lower MSE) by trading a little bias for much less variance.
2. The Bayesian classifier is more accurate **and** much better calibrated at small *n*; MLE is over-confident.
3. The gap grows with dimensionality.
4. With enough data, both converge to the optimal classifier.
5. The prior is a double-edged sword: informative when right, harmful when strongly wrong.

In [ ]:
# Optional: download all figures as a zip
import shutil
shutil.make_archive("results", "zip", OUT)
try:
    from google.colab import files; files.download("results.zip")
except ImportError:
    print("Not on Colab - results.zip saved locally")